In [119]:
import subprocess
import sys

def install_packages():
    """Install required packages silently"""
    packages = [
        "python-docx",
        "pandas",
        "openai",
        "chromadb",
        "faiss-cpu",
        "sentence-transformers",
        "nltk",
        "tiktoken",
        "numpy"
    ]

    for package in packages:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
            print(f"✓ {package} installed")
        except Exception as e:
            print(f"✗ {package} failed: {e}")

install_packages()
print("\n" + "="*60)
print("✓ All packages installed successfully!")
print("="*60 + "\n")



✓ python-docx installed
✓ pandas installed
✓ openai installed
✓ chromadb installed
✓ faiss-cpu installed
✓ sentence-transformers installed
✓ nltk installed
✓ tiktoken installed
✓ numpy installed

✓ All packages installed successfully!



In [120]:
import re
import time
import json
import numpy as np
import pandas as pd
from pathlib import Path
from docx import Document
from typing import List, Dict, Tuple, Any, Optional
import tiktoken
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sentence_transformers import SentenceTransformer
import chromadb
import faiss
import openai
from google import genai

# === DOWNLOAD NLTK DATA ===
print("Downloading NLTK data...")
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
print("✓ NLTK data downloaded\n")


✓ NLTK data downloaded



In [121]:
!pip install -q google-generativeai sentence-transformers tiktoken
!pip install -q google-generativeai sentence-transformers nltk


In [122]:
!pip install -q google-genai

In [123]:
# === IMPORTS ===
import google.generativeai as genai
import os
from nltk.stem import WordNetLemmatizer
from sentence_transformers import SentenceTransformer
from google import genai

# === CONFIGURE GEMINI ===
from google import genai

# Configure client
client = genai.Client(api_key="YOUR_API_KEY")

print("✓ Gemini client initialized")



# === INITIALIZE LEMMATIZER ===
lemmatizer = WordNetLemmatizer()
print("✓ Lemmatizer initialized")

# === INITIALIZE SENTENCE TRANSFORMER ===
print("Loading MiniLM model (384-dim embeddings, free local)...")
minilm_model = SentenceTransformer("all-MiniLM-L6-v2")
print("✓ MiniLM model loaded (dimension: 384)")

print("="*60)
print("✓✓✓ ALL SETUP COMPLETE ✓✓✓")
print("="*60)

✓ Gemini client initialized
✓ Lemmatizer initialized
Loading MiniLM model (384-dim embeddings, free local)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ MiniLM model loaded (dimension: 384)
✓✓✓ ALL SETUP COMPLETE ✓✓✓


In [124]:
import os
import re
import time
import json
import numpy as np
import pandas as pd
from pathlib import Path
from docx import Document
from typing import List, Dict, Tuple, Any, Optional
import tiktoken
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sentence_transformers import SentenceTransformer
import chromadb
import faiss
import openai
from openai import OpenAI
# Create sample data directory
data_dir = Path("sample_data")
data_dir.mkdir(exist_ok=True)

print("Creating sample DOCX (HR Policies)...")

doc = Document()
doc.add_heading("NovaTech Inc. - HR Policy Handbook", 0)

doc.add_heading("1. Annual Leave Policy", level=1)
doc.add_paragraph(
    "All full-time employees are entitled to 20 days of paid annual leave per year. "
    "This includes public holidays. Employees must request leave at least 2 weeks in advance "
    "through the HR portal. Unused leave can be carried over to the next year up to a maximum "
    "of 5 days. Sick leave is separate and should be reported to your manager on the day."
)

doc.add_heading("2. Work-From-Home Policy", level=1)
doc.add_paragraph(
    "Starting from 2024, all employees can work from home up to 3 days per week. "
    "Employees must coordinate with their team leads. Core collaboration hours are 10 AM to 3 PM. "
    "Company equipment must be secured and all confidential data must follow data protection guidelines. "
    "Home office setup is reimbursed up to $500."
)

doc.add_heading("3. Health & Wellness Benefits", level=1)
doc.add_paragraph(
    "Company provides: (a) Comprehensive health insurance for employee and family; "
    "(b) Mental health support via EAP (Employee Assistance Program); "
    "(c) Gym membership reimbursement up to $50/month; "
    "(d) Annual health checkup; (e) Parental leave of 4 months (paid) for both parents."
)

doc.add_heading("4. Professional Development", level=1)
doc.add_paragraph(
    "Every employee receives $2000 annually for training, certifications, or conferences. "
    "Tuition reimbursement for relevant degree programs up to 50%. "
    "Internal mentorship program available. Career development plans reviewed quarterly."
)

doc.save(str(data_dir / "hr_policies.docx"))
print("✓ Created: sample_data/hr_policies.docx\n")


Creating sample DOCX (HR Policies)...
✓ Created: sample_data/hr_policies.docx



In [125]:
print("Creating sample CSV (Product FAQ)...")

faq_data = {
    "question": [
        "What is NovaTech CloudSync?",
        "How do I set up CloudSync on my computer?",
        "What is the pricing model for CloudSync?",
        "Is my data encrypted?",
        "Can I sync multiple devices?",
        "What file formats does CloudSync support?",
        "How much storage do I get?",
        "Do you offer customer support?",
        "Can I integrate CloudSync with other apps?",
        "What happens if I exceed my storage quota?",
    ],
    "answer": [
        "CloudSync is a cloud-based file synchronization and collaboration platform that securely syncs files across multiple devices in real-time.",
        "Download the CloudSync application from our website, install it, log in with your account, and select folders to sync. Setup takes less than 5 minutes.",
        "CloudSync offers three plans: Free (5GB), Pro ($9.99/month for 1TB), and Business ($19.99/month for 5TB). Annual plans get 20% discount.",
        "Yes, all data is encrypted with AES-256 encryption both in transit and at rest. We use industry-standard TLS 1.3 for all connections.",
        "Absolutely! You can sync CloudSync across up to 5 devices on the Free plan and unlimited devices on Pro/Business plans.",
        "CloudSync supports all file formats: documents, images, videos, archives, and more. No restrictions on file type.",
        "Free plan includes 5GB. Pro plan includes 1TB (1000GB). Business plan includes 5TB (5000GB). You can purchase additional storage at $0.99 per 100GB.",
        "Yes, we offer 24/7 email support (response within 1 hour for Pro/Business). Premium phone support available for Business plans.",
        "Yes! CloudSync integrates with Slack, Microsoft Teams, Zapier, and IFTTT. API documentation is available for custom integrations.",
        "When you exceed quota, sync pauses. You can either purchase additional storage or delete old files to continue syncing.",
    ],
    "category": [
        "Products", "Getting Started", "Pricing", "Security", "Sync",
        "Features", "Storage", "Support", "Integration", "Billing"
    ]
}

df_faq = pd.DataFrame(faq_data)
df_faq.to_csv(str(data_dir / "product_faq.csv"), index=False, encoding="utf-8")
print("✓ Created: sample_data/product_faq.csv\n")


Creating sample CSV (Product FAQ)...
✓ Created: sample_data/product_faq.csv



In [126]:
print("Creating sample TXT (Financial Summary)...")

txt_content = """
NovaTech Inc. - Financial Summary Q3 2024
==========================================

REVENUE & GROWTH
─────────────────

Total Revenue (Q3 2024): $12.5 Million
Year-over-Year Growth: 42%
Quarter-over-Quarter Growth: 18%

Revenue by Product:
- CloudSync (File Sync): $8.2 Million (65.6%)
- DataVault (Analytics): $3.1 Million (24.8%)
- SecureShield (Security): $1.2 Million (9.6%)

EXPENSES & PROFITABILITY
─────────────────────────

Operating Expenses: $7.8 Million
- Engineering & R&D: $3.5 Million (44.9%)
- Sales & Marketing: $2.1 Million (26.9%)
- Operations & Admin: $1.5 Million (19.2%)
- Infrastructure & Support: $0.7 Million (9.0%)

Operating Profit: $4.7 Million
Operating Margin: 37.6%
Net Profit: $3.9 Million
Net Margin: 31.2%

CUSTOMER METRICS
─────────────────

Total Active Users: 285,000
Enterprise Customers: 520 (avg contract value: $45K/year)
SMB Customers: 15,847 (avg contract value: $1.2K/year)
Customer Retention Rate: 96.3%
Monthly Churn Rate: 0.8%

CASH POSITION
──────────────

Cash & Equivalents: $28.4 Million
Burn Rate: Negative (profitable)
Runway: 45+ years (self-sufficient)

FORWARD OUTLOOK
─────────────────

Q4 2024 Revenue Forecast: $13.8 Million
FY 2024 Revenue Target: $50 Million
Expected Headcount by EOY: 320 employees (currently 185)
New Product Launch Timeline: Q2 2025 (AI-Powered Data Insights)

KEY ACHIEVEMENTS
──────────────────

- Launched CloudSync 3.0 with AI recommendation engine
- Expanded to 4 new geographic markets (Japan, Germany, Brazil, Australia)
- SOC 2 Type II certification achieved
- Mobile app reached 2 million downloads
- 5 new enterprise partnerships signed with Fortune 500 companies
"""

with open(data_dir / "financial_summary.txt", "w", encoding="utf-8") as f:
    f.write(txt_content)
print("✓ Created: sample_data/financial_summary.txt\n")

print("="*60)
print("✓ All sample files generated successfully!")
print("="*60 + "\n")


Creating sample TXT (Financial Summary)...
✓ Created: sample_data/financial_summary.txt

✓ All sample files generated successfully!



In [127]:
def load_txt(file_path: str) -> List[Dict[str, Any]]:
    """
    📌 Same as Day 1: Load a plain text file.
    Returns a list with single document dict.
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    return [{
        "content": content,
        "metadata": {
            "source": Path(file_path).name,
            "type": "txt"
        }
    }]




In [128]:

def load_docx(file_path: str) -> List[Dict[str, Any]]:

    doc = Document(file_path)

    # Extract all paragraphs (skip empty ones)
    content = "\n\n".join([p.text for p in doc.paragraphs if p.text.strip()])

    return [{
        "content": content,
        "metadata": {
            "source": Path(file_path).name,
            "type": "docx",
            "total_paragraphs": len([p for p in doc.paragraphs if p.text.strip()])
        }
    }]




In [129]:
def load_csv(file_path: str, text_columns: Optional[List[str]] = None) -> List[Dict[str, Any]]:

    df = pd.read_csv(file_path)

    documents = []
    for idx, row in df.iterrows():
        # If specific columns requested, use only those; else use all
        if text_columns:
            content = " ".join([str(row[col]) for col in text_columns if col in row.index])
        else:
            content = " ".join([str(val) for val in row.values])

        documents.append({
            "content": content,
            "metadata": {
                "source": Path(file_path).name,
                "type": "csv",
                "row_number": idx,
                "columns": list(df.columns)
            }
        })

    return documents

In [130]:
pip install pypdf

In [131]:
def load_pdf(file_path: str) -> List[Dict[str, Any]]:

    # In production: use pypdf, pdfplumber, or similar
    # For now, just return empty list
    print(f"Note: PDF loader would load {file_path} (pypdf not installed)")
    return []


In [132]:
def load_documents(file_paths: List[str]) -> Tuple[List[Dict[str, Any]], Dict[str, int]]:

    all_documents = []
    file_type_counts = {}

    # Map extensions to loader functions
    loaders = {
        ".txt": load_txt,
        ".docx": load_docx,
        ".csv": load_csv,
        ".pdf": load_pdf,
    }

    for file_path in file_paths:
        # Skip if file doesn't exist
        if not Path(file_path).exists():
            print(f"⚠️  Skipping {file_path} (not found)")
            continue

        # Get file extension
        ext = Path(file_path).suffix.lower()

        # Get appropriate loader
        loader = loaders.get(ext)
        if not loader:
            print(f"⚠️  Skipping {file_path} (unsupported format: {ext})")
            continue

        # Load documents
        try:
            docs = loader(file_path)
            all_documents.extend(docs)

            # Track counts
            file_type = ext.replace(".", "").upper()
            file_type_counts[file_type] = file_type_counts.get(file_type, 0) + len(docs)
            print(f"✓ Loaded {len(docs)} documents from {Path(file_path).name}")

        except Exception as e:
            print(f"✗ Error loading {file_path}: {e}")

    return all_documents, file_type_counts


In [133]:
file_paths = [
    str(data_dir / "hr_policies.docx"),
    str(data_dir / "product_faq.csv"),
    str(data_dir / "financial_summary.txt"),
]

all_documents, type_counts = load_documents(file_paths)

print("\n" + "="*60)
print("📋 DOCUMENT LOADING SUMMARY")
print("="*60)
print(f"Total documents loaded: {len(all_documents)}")
print(f"Files by type: {type_counts}")
print("\nFirst document (first 300 chars):")
print(f"  Source: {all_documents[0]['metadata']['source']}")
print(f"  Content: {all_documents[0]['content'][:300]}...")
print("="*60 + "\n")


✓ Loaded 1 documents from hr_policies.docx
✓ Loaded 10 documents from product_faq.csv
✓ Loaded 1 documents from financial_summary.txt

📋 DOCUMENT LOADING SUMMARY
Total documents loaded: 12
Files by type: {'DOCX': 1, 'CSV': 10, 'TXT': 1}

First document (first 300 chars):
  Source: hr_policies.docx
  Content: NovaTech Inc. - HR Policy Handbook

1. Annual Leave Policy

All full-time employees are entitled to 20 days of paid annual leave per year. This includes public holidays. Employees must request leave at least 2 weeks in advance through the HR portal. Unused leave can be carried over to the next year ...



In [134]:
def count_tokens(text: str) -> Tuple[int, int]:
    """
    Count characters and estimate tokens for Gemini models.
    If Gemini token counting is available, use it.
    Otherwise fall back to an approximate estimate.
    """

    char_count = len(text)

    try:
        token_info = model.count_tokens(text)
        token_count = token_info.total_tokens
    except Exception:
        # fallback estimate: ~4 characters per token
        token_count = max(1, char_count // 4)

    return char_count, token_count

In [135]:
def split_sentences(text: str) -> List[str]:

    return sent_tokenize(text)


In [136]:
def remove_stopwords(text: str) -> str:

    stop_words = set(stopwords.words('english'))
    tokens = word_tokenize(text.lower())
    filtered = [w for w in tokens if w not in stop_words and w.isalnum()]
    return " ".join(filtered)


In [137]:
nltk.download("averaged_perceptron_tagger")

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [138]:
from nltk.corpus import wordnet
from nltk import pos_tag
from nltk.tokenize import word_tokenize

def get_wordnet_pos(tag):
    if tag.startswith("J"):
        return wordnet.ADJ
    elif tag.startswith("V"):
        return wordnet.VERB
    elif tag.startswith("N"):
        return wordnet.NOUN
    elif tag.startswith("R"):
        return wordnet.ADV
    else:
        return wordnet.NOUN

In [139]:
def lemmatize_text(text: str) -> str:
    tokens = word_tokenize(text.lower())
    pos_tags = pos_tag(tokens)

    lemmas = [
        lemmatizer.lemmatize(word, get_wordnet_pos(tag))
        for word, tag in pos_tags
    ]

    return " ".join(lemmas)

In [140]:
def clean_basic(text: str) -> str:

    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


In [141]:
import nltk

nltk.download("punkt")
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("averaged_perceptron_tagger_eng")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [142]:
def preprocess_full(text: str, use_lemmatization: bool = True) -> str:

    # Step 1: Basic cleaning
    text = clean_basic(text)

    # Step 2: Optional lemmatization
    if use_lemmatization:
        text = lemmatize_text(text)

    # Step 3: Stop-word removal (conservative for documents)
    # Note: We keep stop words in most cases because they carry context
    # Remove them only for keyword matching

    return text


sample_text = all_documents[0]['content'][:500]

print("="*60)
print("📊 TEXT PREPROCESSING DEMONSTRATION")
print("="*60)

# Tokenization
chars, tokens = count_tokens(sample_text)
print(f"\n1️⃣  TOKENIZATION")
print(f"  Characters: {chars}")
print(f"  Tokens (Gemini estimate): {tokens}")
print(f"  Ratio: {chars/tokens:.2f} chars per token\n")

# Sentence splitting
sentences = split_sentences(sample_text)
print(f"2️⃣  SENTENCE SPLITTING")
for i, sent in enumerate(sentences[:3], 1):
    print(f"  [{i}] {sent}\n")

# Lemmatization demo
sample_sentence = "The developers are running and building better systems."
lemmatized = lemmatize_text(sample_sentence)
print(f"3️⃣  LEMMATIZATION")
print(f"  Original:  {sample_sentence}")
print(f"  Lemmatized: {lemmatized}\n")

# Full preprocessing
print(f"4️⃣  FULL PREPROCESSING PIPELINE")
print(f"  Original (first 200 chars): {sample_text[:200]}")
preprocessed = preprocess_full(sample_text)
print(f"  Preprocessed (first 200 chars): {preprocessed[:200]}\n")

print("="*60 + "\n")



📊 TEXT PREPROCESSING DEMONSTRATION



1️⃣  TOKENIZATION
  Characters: 500
  Tokens (Gemini estimate): 125
  Ratio: 4.00 chars per token

2️⃣  SENTENCE SPLITTING
  [1] NovaTech Inc. - HR Policy Handbook

1.

  [2] Annual Leave Policy

All full-time employees are entitled to 20 days of paid annual leave per year.

  [3] This includes public holidays.

3️⃣  LEMMATIZATION
  Original:  The developers are running and building better systems.
  Lemmatized: the developer be run and building good system .

4️⃣  FULL PREPROCESSING PIPELINE
  Original (first 200 chars): NovaTech Inc. - HR Policy Handbook

1. Annual Leave Policy

All full-time employees are entitled to 20 days of paid annual leave per year. This includes public holidays. Employees must request leave a
  Preprocessed (first 200 chars): novatech inc. - hr policy handbook 1. annual leave policy all full-time employee be entitle to 20 day of paid annual leave per year . this include public holiday . employee must request leave at least




In [143]:
def strategy_1_sentence_chunk(text: str, n_sentences: int = 3) -> List[str]:
    sentences = split_sentences(text)
    chunks = [" ".join(sentences[i:i+n_sentences])
              for i in range(0, len(sentences), n_sentences)]
    return chunks



In [144]:
def strategy_2_paragraph_chunk(text: str) -> List[str]:

    # Split on double newlines
    paragraphs = text.split('\n\n')
    # Filter empty paragraphs
    chunks = [p.strip() for p in paragraphs if p.strip()]
    return chunks


In [145]:
def strategy_3_token_chunk(text: str, max_tokens: int = 200) -> List[str]:

    tokens = encoding.encode(text)
    chunks = []

    for i in range(0, len(tokens), max_tokens):
        chunk_tokens = tokens[i:i+max_tokens]
        # Decode back to text
        chunk_text = encoding.decode(chunk_tokens)
        if chunk_text.strip():
            chunks.append(chunk_text)

    return chunks


In [146]:
def strategy_4_header_chunk(text: str) -> List[Tuple[str, str]]:

    # Split on markdown headers or numbered sections
    pattern = r'^(#{1,3} .+|^\d+\. .+)$'  # Markdown headers or numbered sections

    lines = text.split('\n')
    sections = []
    current_header = "Untitled"
    current_content = []

    for line in lines:
        if re.match(pattern, line, re.MULTILINE):
            # Save previous section
            if current_content:
                sections.append((current_header, "\n".join(current_content).strip()))
            current_header = line.strip()
            current_content = []
        else:
            current_content.append(line)

    # Save last section
    if current_content:
        sections.append((current_header, "\n".join(current_content).strip()))

    return sections


In [147]:
def strategy_5_recursive_chunk(text: str, chunk_size: int = 200, overlap: int = 20) -> List[str]:

    # Get sentences (natural boundaries)
    sentences = split_sentences(text)

    chunks = []
    current_chunk = []
    current_tokens = 0
    overlap_buffer = []

    for sentence in sentences:
        sentence_tokens = len(encoding.encode(sentence))

        # If adding sentence exceeds chunk size, save chunk and start new
        if current_tokens + sentence_tokens > chunk_size and current_chunk:
            # Save chunk
            chunk_text = " ".join(current_chunk)
            chunks.append(chunk_text)

            # Prepare overlap: keep last N tokens
            overlap_tokens = len(encoding.encode(" ".join(current_chunk)))
            overlap_kept = min(overlap, overlap_tokens)

            # Start new chunk with overlap
            if overlap_kept > 0:
                overlap_buffer = current_chunk[max(0, len(current_chunk)-1):]
            else:
                overlap_buffer = []

            current_chunk = overlap_buffer
            current_tokens = sum(len(encoding.encode(s)) for s in current_chunk)

        # Add sentence
        current_chunk.append(sentence)
        current_tokens += sentence_tokens

    # Don't forget last chunk
    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks


In [148]:
def compare_chunking_strategies(text: str) -> None:
    """
    🆕 Compare all 5 chunking strategies on sample text.
    Shows metrics for each strategy.
    """
    print("="*80)
    print("🔪 COMPARING 5 CHUNKING STRATEGIES")
    print("="*80 + "\n")

    # Strategy 1: Sentence
    print("1️⃣  SENTENCE-BASED CHUNKING (3 sentences/chunk)")
    chunks1 = strategy_1_sentence_chunk(text, n_sentences=3)
    print(f"   Total chunks: {len(chunks1)}")
    print(f"   Avg chunk size: {len(text)//len(chunks1) if chunks1 else 0} chars")
    print(f"   Sample: {chunks1[0][:100]}...\n")

    # Strategy 2: Paragraph
    print("2️⃣  PARAGRAPH-BASED CHUNKING (split on \\n\\n)")
    chunks2 = strategy_2_paragraph_chunk(text)
    print(f"   Total chunks: {len(chunks2)}")
    print(f"   Avg chunk size: {len(text)//len(chunks2) if chunks2 else 0} chars")
    if chunks2:
        print(f"   Sample: {chunks2[0][:100]}...\n")

    # Strategy 3: Token
    print("3️⃣  TOKEN-BASED CHUNKING (200 tokens/chunk)")
    chunks3 = strategy_3_token_chunk(text, max_tokens=200)
    print(f"   Total chunks: {len(chunks3)}")
    print(f"   Avg chunk size: {len(text)//len(chunks3) if chunks3 else 0} chars")
    print(f"   Sample: {chunks3[0][:100]}...\n")

    # Strategy 4: Header
    print("4️⃣  HEADER-BASED CHUNKING")
    chunks4 = strategy_4_header_chunk(text)
    print(f"   Total sections: {len(chunks4)}")
    if chunks4:
        print(f"   Headers found: {[h for h, _ in chunks4[:3]]}")
        print(f"   Sample: {chunks4[0][1][:100]}...\n")

    # Strategy 5: Recursive
    print("5️⃣  RECURSIVE CHUNKING (200 tokens, 20 overlap)")
    chunks5 = strategy_5_recursive_chunk(text, chunk_size=200, overlap=20)
    print(f"   Total chunks: {len(chunks5)}")
    print(f"   Avg chunk size: {len(text)//len(chunks5) if chunks5 else 0} chars")
    print(f"   Sample: {chunks5[0][:100]}...\n")

    # Tuning comparison for Strategy 5
    print("5️⃣  RECURSIVE TUNING TABLE")
    print("   Size | Overlap | Chunks | Avg chars")
    print("   " + "-"*40)
    for size in [100, 200, 300, 500]:
        for ovlp in [0, 10, 20]:
            chunks = strategy_5_recursive_chunk(text, chunk_size=size, overlap=ovlp)
            avg_size = len(text) // len(chunks) if chunks else 0
            print(f"   {size:3d}  |  {ovlp:2d}%   | {len(chunks):6d} | {avg_size:7d}")

    print("\n" + "="*80)
    print("✓ RECOMMENDATION: Use SENTENCE-BASED chunking (Strategy 1)")
    print("  Reason: Good balance of coherence vs practical chunk sizes")
    print("="*80 + "\n")



In [149]:
def create_chunks_from_documents(documents: List[Dict],
                                chunk_method: str = "sentence",
                                n_sentences: int = 3) -> List[Dict]:
    """
    Apply chosen chunking strategy to all documents.
    Returns chunks as dicts with content and metadata.
    """
    all_chunks = []
    global_chunk_id = 0

    for doc in documents:
        content = doc["content"]
        source = doc["metadata"]["source"]
        doc_type = doc["metadata"]["type"]

        # Apply chosen chunking strategy
        if chunk_method == "sentence":
            raw_chunks = strategy_1_sentence_chunk(content, n_sentences=n_sentences)
        elif chunk_method == "paragraph":
            raw_chunks = strategy_2_paragraph_chunk(content)
        elif chunk_method == "token":
            raw_chunks = strategy_3_token_chunk(content, max_tokens=200)
        elif chunk_method == "recursive":
            raw_chunks = strategy_5_recursive_chunk(content, chunk_size=200, overlap=20)
        else:
            raw_chunks = [content]  # Fallback: single chunk per doc

        # Convert to structured chunks
        for chunk_idx, chunk_content in enumerate(raw_chunks):
            if chunk_content.strip():  # Skip empty chunks
                all_chunks.append({
                    "content": chunk_content,
                    "metadata": {
                        "source": source,
                        "type": doc_type,
                        "chunk_id": global_chunk_id,
                        "chunk_method": chunk_method,
                        "chunk_num": chunk_idx
                    }
                })
                global_chunk_id += 1

    return all_chunks


# Create chunks using sentence-based strategy (best balance)
chunks = create_chunks_from_documents(all_documents, chunk_method="sentence", n_sentences=3)

print("="*60)
print("📦 CHUNKING COMPLETE")
print("="*60)
print(f"Total chunks created: {len(chunks)}")
print(f"Chunks by source:")
for source in set(c['metadata']['source'] for c in chunks):
    count = sum(1 for c in chunks if c['metadata']['source'] == source)
    print(f"  • {source}: {count} chunks")
print(f"\nSample chunk (first 200 chars):\n  {chunks[0]['content'][:200]}...")
print("="*60 + "\n")


📦 CHUNKING COMPLETE
Total chunks created: 27
Chunks by source:
  • hr_policies.docx: 7 chunks
  • product_faq.csv: 19 chunks
  • financial_summary.txt: 1 chunks

Sample chunk (first 200 chars):
  NovaTech Inc. - HR Policy Handbook

1. Annual Leave Policy

All full-time employees are entitled to 20 days of paid annual leave per year. This includes public holidays....



In [150]:
def get_gemini_embeddings(texts: List[str], model: str = "models/text-embedding-004") -> np.ndarray:


    if not api_key:
        print("⚠️ Gemini API key not configured, skipping embeddings")
        return np.zeros((len(texts), 768))

    try:
        start = time.time()

        embeddings = []
        for text in texts:
            response = genai.embed_content(
                model=model,
                content=text,
                task_type="retrieval_document"
            )
            embeddings.append(response["embedding"])

        embeddings = np.array(embeddings)

        elapsed = time.time() - start
        print(f"✓ Gemini embeddings: {len(texts)} texts in {elapsed:.2f}s")

        return embeddings

    except Exception as e:
        print(f"✗ Gemini embedding error: {e}")
        return np.zeros((len(texts), 768))

In [151]:

def get_minilm_embeddings(texts: List[str]) -> np.ndarray:

    start = time.time()
    embeddings = minilm_model.encode(texts, show_progress_bar=False)
    elapsed = time.time() - start
    print(f"✓ MiniLM embeddings: {len(texts)} texts in {elapsed:.2f}s")
    return np.array(embeddings)


In [152]:
# === GENERATE EMBEDDINGS FOR ALL CHUNKS ===

print("="*60)
print("🧠 GENERATING EMBEDDINGS")
print("="*60 + "\n")

# Extract chunk texts
chunk_texts = [c["content"] for c in chunks]

# ---------------------------------------------------
# 1️⃣ MiniLM embeddings (PRIMARY - local, free)
# ---------------------------------------------------
print("Encoding with MiniLM (384-dim, local, free)...")

minilm_embeddings = get_minilm_embeddings(chunk_texts)

print(f"  Shape: {minilm_embeddings.shape}")
print(f"  Dimension: {minilm_embeddings.shape[1]}\n")


# ---------------------------------------------------
# 2️⃣ Gemini embeddings (API - demonstration only)
# ---------------------------------------------------
print("Encoding with Gemini embeddings (768-dim, API)...")

try:
    gemini_embeddings = get_gemini_embeddings(chunk_texts[:5])  # small sample

    print(f"  Shape (demo): {gemini_embeddings.shape}")
    print(f"  Dimension: {gemini_embeddings.shape[1]}\n")

except Exception as e:
    print(f"  ⚠️ Gemini embedding failed: {e}")
    gemini_embeddings = None


# ---------------------------------------------------
# 3️⃣ Comparison
# ---------------------------------------------------
print("COMPARISON:")

print(
    f"  MiniLM:  {minilm_embeddings.shape[1]} dims | "
    f"{minilm_embeddings.shape[0]} chunks | "
    f"Cost: $0 | Speed: Fast | Local model"
)

if gemini_embeddings is not None:
    print(
        f"  Gemini:  {gemini_embeddings.shape[1]} dims | "
        f"{gemini_embeddings.shape[0]} demo chunks | "
        f"Cost: API usage | Speed: Medium"
    )

print("  → Using MiniLM as DEFAULT (free + fast!)")

print("\n" + "="*60 + "\n")

🧠 GENERATING EMBEDDINGS

Encoding with MiniLM (384-dim, local, free)...
✓ MiniLM embeddings: 27 texts in 0.10s
  Shape: (27, 384)
  Dimension: 384

Encoding with Gemini embeddings (768-dim, API)...
✗ Gemini embedding error: module 'google.genai' has no attribute 'embed_content'
  Shape (demo): (5, 768)
  Dimension: 768

COMPARISON:
  MiniLM:  384 dims | 27 chunks | Cost: $0 | Speed: Fast | Local model
  Gemini:  768 dims | 5 demo chunks | Cost: API usage | Speed: Medium
  → Using MiniLM as DEFAULT (free + fast!)




In [153]:
# === VECTOR STORES ===

print("="*60)
print("🗂️  INITIALIZING VECTOR STORES")
print("="*60 + "\n")

# ====== 1. CHROMADB ======
print("1️⃣  Setting up ChromaDB...")
chroma_client = chromadb.EphemeralClient()  # In-memory for this demo
chroma_collection = chroma_client.get_or_create_collection(
    name="rag_chunks",
    metadata={"hnsw:space": "cosine"}  # Use cosine distance
)

# Add chunks with MiniLM embeddings and metadata
chroma_ids = [f"chunk_{i}" for i in range(len(chunks))]
chroma_collection.add(
    ids=chroma_ids,
    embeddings=minilm_embeddings.tolist(),  # Convert to list for ChromaDB
    documents=[c["content"] for c in chunks],
    metadatas=[c["metadata"] for c in chunks]
)
print(f"✓ ChromaDB: Stored {chroma_collection.count()} chunks\n")


🗂️  INITIALIZING VECTOR STORES

1️⃣  Setting up ChromaDB...
✓ ChromaDB: Stored 27 chunks



In [154]:
print("2️⃣  Setting up FAISS...")

# Create FAISS index
dimension = minilm_embeddings.shape[1]  # 384 for MiniLM
faiss_index = faiss.IndexFlatL2(dimension)  # L2 distance (Euclidean)

# Convert embeddings to float32 (required by FAISS)
faiss_embeddings = np.array(minilm_embeddings, dtype="float32")
faiss_index.add(faiss_embeddings)

# Create metadata lookup (FAISS doesn't store metadata)
faiss_metadata_lookup = {i: chunks[i]["metadata"] for i in range(len(chunks))}
faiss_id_lookup = {i: f"chunk_{i}" for i in range(len(chunks))}

print(f"✓ FAISS: Stored {faiss_index.ntotal} vectors")
print(f"  Dimension: {dimension}")
print(f"  Memory: ~{faiss_index.ntotal * dimension * 4 / (1024*1024):.2f} MB\n")


2️⃣  Setting up FAISS...
✓ FAISS: Stored 27 vectors
  Dimension: 384
  Memory: ~0.04 MB



In [155]:
print("VECTOR STORE COMPARISON:")
print(f"  ChromaDB: {chroma_collection.count()} vectors | Easy metadata | Good for demos")
print(f"  FAISS:    {faiss_index.ntotal} vectors | Pure speed | Manual metadata")

print("\n" + "="*60 + "\n")

VECTOR STORE COMPARISON:
  ChromaDB: 27 vectors | Easy metadata | Good for demos
  FAISS:    27 vectors | Pure speed | Manual metadata




In [156]:
def score_to_similarity(distance: float, metric: str = "l2") -> float:

    if metric == "l2":
        # L2 distance to similarity: exp(-distance)
        return np.exp(-distance)
    else:
        return 1 / (1 + distance)


In [157]:
def retrieve_from_chromadb(query: str, k: int = 5,
                          source_filter: Optional[str] = None) -> List[Dict]:

    # Embed query
    query_embedding = minilm_model.encode([query])[0].tolist()

    # Build where filter if needed
    where_filter = None
    if source_filter:
        where_filter = {"source": {"$eq": source_filter}}

    # Query ChromaDB
    results = chroma_collection.query(
        query_embeddings=[query_embedding],
        n_results=k,
        where=where_filter
    )

    # Format results
    formatted = []
    if results["ids"] and len(results["ids"]) > 0:
        for i, chunk_id in enumerate(results["ids"][0]):
            # ChromaDB returns distances; convert to similarity
            distance = results["distances"][0][i] if "distances" in results else 0
            similarity = score_to_similarity(distance, metric="l2")

            # Find the original chunk
            chunk_idx = int(chunk_id.split("_")[1])
            formatted.append({
                "content": chunks[chunk_idx]["content"],
                "metadata": chunks[chunk_idx]["metadata"],
                "score": similarity,
                "distance": distance
            })

    return formatted



In [158]:
def retrieve_from_chromadb(query: str, k: int = 5,
                          source_filter: Optional[str] = None) -> List[Dict]:

    # Embed query
    query_embedding = minilm_model.encode([query])[0].tolist()

    # Build where filter if needed
    where_filter = None
    if source_filter:
        where_filter = {"source": {"$eq": source_filter}}

    # Query ChromaDB
    results = chroma_collection.query(
        query_embeddings=[query_embedding],
        n_results=k,
        where=where_filter
    )

    # Format results
    formatted = []
    if results["ids"] and len(results["ids"]) > 0:
        for i, chunk_id in enumerate(results["ids"][0]):
            # ChromaDB returns distances; convert to similarity
            distance = results["distances"][0][i] if "distances" in results else 0
            similarity = score_to_similarity(distance, metric="l2")

            # Find the original chunk
            chunk_idx = int(chunk_id.split("_")[1])
            formatted.append({
                "content": chunks[chunk_idx]["content"],
                "metadata": chunks[chunk_idx]["metadata"],
                "score": similarity,
                "distance": distance
            })

    return formatted



In [159]:
def retrieve_with_threshold(query: str, min_score: float = 0.7,
                           k: int = 10, source_filter: Optional[str] = None) -> List[Dict]:

    # Get initial results
    results = retrieve_from_chromadb(query, k=k, source_filter=source_filter)

    # Filter by score
    filtered = [r for r in results if r["score"] >= min_score]

    return filtered



In [160]:
def smart_retrieve(query: str, k: int = 10, rerank_top: int = 3,
                   min_score: float = 0.7, source_filter: Optional[str] = None) -> List[Dict]:

    # Step 1 & 2: Retrieve with optional filtering
    results = retrieve_from_chromadb(query, k=k, source_filter=source_filter)

    # Step 3: Apply score threshold
    results = [r for r in results if r["score"] >= min_score]

    # Step 4: Re-rank (sort by score, descending)
    results = sorted(results, key=lambda x: x["score"], reverse=True)

    # Keep top N
    results = results[:rerank_top]

    return results


In [161]:
print("="*60)
print("🔍 TESTING RETRIEVAL FUNCTIONS")
print("="*60 + "\n")

# Test query
test_query = "What are the remote work policies?"

print(f"Query: {test_query}\n")

# Test with different K values
print("Testing K-tuning (retrieve top-K chunks):")
for k_val in [1, 3, 5, 10]:
    results = retrieve_from_chromadb(test_query, k=k_val)
    print(f"  K={k_val}: Retrieved {len(results)} chunks, max score: {results[0]['score']:.3f}")

print("\nTesting score threshold filtering:")
results_full = retrieve_from_chromadb(test_query, k=10)
for threshold in [0.5, 0.65, 0.8]:
    filtered = [r for r in results_full if r["score"] >= threshold]
    print(f"  Min score {threshold:.2f}: {len(filtered)}/{len(results_full)} chunks kept")

print("\n" + "="*60 + "\n")


🔍 TESTING RETRIEVAL FUNCTIONS

Query: What are the remote work policies?

Testing K-tuning (retrieve top-K chunks):
  K=1: Retrieved 1 chunks, max score: 0.579
  K=3: Retrieved 3 chunks, max score: 0.579
  K=5: Retrieved 5 chunks, max score: 0.579
  K=10: Retrieved 10 chunks, max score: 0.579

Testing score threshold filtering:
  Min score 0.50: 4/10 chunks kept
  Min score 0.65: 0/10 chunks kept
  Min score 0.80: 0/10 chunks kept




In [162]:
def rerank_results(results: List[Dict], top_n: int = 3, min_score: float = 0.6) -> List[Dict]:
    """
    🆕 NEW: Re-rank retrieved results by score.

    Steps:
    1. Sort by score (highest first)
    2. Filter out results below min_score
    3. Keep only top N results

    Args:
        results: List of result dicts with 'score' key
        top_n: Number of top results to keep
        min_score: Minimum score threshold

    Returns:
        Sorted and filtered results
    """
    # Step 1: Sort by score (descending)
    sorted_results = sorted(results, key=lambda x: x["score"], reverse=True)

    # Step 2: Filter by score threshold
    filtered_results = [r for r in sorted_results if r["score"] >= min_score]

    # Step 3: Keep top N
    reranked = filtered_results[:top_n]

    return reranked


In [163]:
print("="*60)
print("📊 RE-RANKING DEMONSTRATION")
print("="*60 + "\n")

# Retrieve top-10 (retrieve broadly)
demo_query = "What is CloudSync?"
results_before = retrieve_from_chromadb(demo_query, k=10)

print(f"Query: {demo_query}\n")
print("BEFORE RE-RANKING (all 10 results):")
print("  Rank | Score | Content (first 60 chars)")
print("  " + "-"*60)
for i, r in enumerate(results_before, 1):
    print(f"  {i:2d}   | {r['score']:.3f} | {r['content'][:60]}...")

# Re-rank: keep top 3 with min score 0.6
results_after = rerank_results(results_before, top_n=3, min_score=0.6)

print(f"\nAFTER RE-RANKING (top 3, min_score=0.6):")
print("  Rank | Score | Content (first 60 chars)")
print("  " + "-"*60)
for i, r in enumerate(results_after, 1):
    print(f"  {i:2d}   | {r['score']:.3f} | {r['content'][:60]}...")

print(f"\n✓ Kept {len(results_after)}/{len(results_before)} chunks (best quality)")

print("\n" + "="*60 + "\n")


📊 RE-RANKING DEMONSTRATION

Query: What is CloudSync?

BEFORE RE-RANKING (all 10 results):
  Rank | Score | Content (first 60 chars)
  ------------------------------------------------------------
   1   | 0.770 | What is NovaTech CloudSync? CloudSync is a cloud-based file ...
   2   | 0.672 | Can I integrate CloudSync with other apps? Yes! CloudSync in...
   3   | 0.667 | How do I set up CloudSync on my computer? Download the Cloud...
   4   | 0.616 | What file formats does CloudSync support? CloudSync supports...
   5   | 0.616 | What is the pricing model for CloudSync? CloudSync offers th...
   6   | 0.559 | Sync...
   7   | 0.555 | Can I sync multiple devices? Absolutely! You can sync CloudS...
   8   | 0.487 | 
NovaTech Inc. - Financial Summary Q3 2024
=================...
   9   | 0.484 | What happens if I exceed my storage quota? When you exceed q...
  10   | 0.435 | API documentation is available for custom integrations. Inte...

AFTER RE-RANKING (top 3, min_score=0.6):
  Rank |

In [164]:
def build_prompt_v2(query: str, results: List[Dict]) -> Tuple[str, str]:


    # === SYSTEM PROMPT (enforces behavior) ===
    system_prompt = """You are a precise, helpful assistant. Follow these rules STRICTLY:

1. Answer ONLY based on the provided context documents [1], [2], [3], etc.
2. Cite your sources clearly using [1], [2], [3] notation in your answer
3. If the context does not contain enough information to answer, say exactly: "I don't have enough information to answer this based on the available documents."
4. Never fabricate, guess, assume, or use knowledge outside the provided context
5. Be concise and direct
6. If the question asks about something not in the documents, refuse to answer"""

    # === USER PROMPT (provides context and question) ===
    context_parts = []
    for i, result in enumerate(results, 1):
        source = result["metadata"]["source"]
        score = result["score"]
        content = result["content"]
        context_parts.append(f"[{i}] (From: {source}, Relevance: {score:.2f})\n{content}")

    context_text = "\n\n".join(context_parts)

    user_prompt = f"""Context documents:
{context_text}

Question: {query}

Please answer the question using ONLY the context provided above.
Cite your sources using [1], [2], [3] notation.
If you cannot answer from the context, say so explicitly."""

    return system_prompt, user_prompt


In [165]:
def call_llm(system_prompt: str, user_prompt: str, temperature: float = 0.2) -> str:

    full_prompt = f"""
{system_prompt}

{user_prompt}
"""

    try:
        response = client.models.generate_content(
            model="gemini-2.0-flash",   # ✅ WORKING MODEL
            contents=full_prompt
        )

        return response.text.strip()

    except Exception as e:
        return f"[Error calling LLM: {e}]"



In [166]:
print("="*60)
print("💬 LLM INTEGRATION DEMO")
print("="*60 + "\n")

demo_query = "What are the leave benefits?"
demo_results = smart_retrieve(demo_query, k=10, rerank_top=3, min_score=0.6)

print(f"Query: {demo_query}")
print(f"Retrieved & Re-ranked: {len(demo_results)} chunks\n")

# Build prompts
system, user = build_prompt_v2(demo_query, demo_results)

print("SYSTEM PROMPT:")
print("-"*60)
print(system)

print("\n\nUSER PROMPT:")
print("-"*60)
print(user[:500] + "...\n")

# Call LLM
print("CALLING GPT-4o-mini...")
answer = call_llm(system, user, temperature=0.2)

print("\nLLM RESPONSE:")
print("-"*60)
print(answer)

print("\n" + "="*60 + "\n")


💬 LLM INTEGRATION DEMO

Query: What are the leave benefits?
Retrieved & Re-ranked: 0 chunks

SYSTEM PROMPT:
------------------------------------------------------------
You are a precise, helpful assistant. Follow these rules STRICTLY:

1. Answer ONLY based on the provided context documents [1], [2], [3], etc.
2. Cite your sources clearly using [1], [2], [3] notation in your answer
3. If the context does not contain enough information to answer, say exactly: "I don't have enough information to answer this based on the available documents."
4. Never fabricate, guess, assume, or use knowledge outside the provided context
5. Be concise and direct
6. If the question asks about something not in the documents, refuse to answer


USER PROMPT:
------------------------------------------------------------
Context documents:


Question: What are the leave benefits?

Please answer the question using ONLY the context provided above.
Cite your sources using [1], [2], [3] notation.
If you cannot answ

In [167]:
def generate_answer_with_gemini(question, context):
    """
    Generate final answer using Gemini LLM
    """

    prompt = f"""
You are a helpful assistant answering questions using provided documents.

Use ONLY the context below to answer.

If the answer is not in the context, respond:
"I cannot find this information in the provided documents."

Context:
{context}

Question:
{question}

Answer:
"""

    try:
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"[Error calling LLM: {e}]"

In [168]:
def rag_pipeline_v2(query: str, k: int = 10, rerank_top: int = 3,
                    min_score: float = 0.6, source_filter: Optional[str] = None) -> Dict:

    # Step 1: Retrieve chunks
    results = smart_retrieve(
        query,
        k=k,
        rerank_top=rerank_top,
        min_score=min_score,
        source_filter=source_filter
    )

    # Handle case where nothing was retrieved
    if not results:
        return {
            "query": query,
            "sources": [],
            "chunks_retrieved": 0,
            "chunks_kept": 0,
            "scores": [],
            "answer": "No relevant information found in the documents.",
            "success": False
        }

    # Step 2: Build prompts
    system_prompt, user_prompt = build_prompt_v2(query, results)

    # Step 3: Generate answer using Gemini
    answer = call_llm(system_prompt, user_prompt, temperature=0.2)

    # Step 4: Collect metadata
    sources = list(set([r["metadata"]["source"] for r in results]))

    scores = [r.get("score", 0) for r in results]

    return {
        "query": query,
        "sources": sources,
        "chunks_retrieved": len(results),
        "chunks_kept": len([s for s in scores if s >= min_score]),
        "scores": scores,
        "answer": answer,
        "success": True
    }

In [169]:
print("="*60)
print("🚀 COMPLETE RAG PIPELINE TEST")
print("="*60 + "\n")

test_query = "How much leave do employees get?"
result = rag_pipeline_v2(test_query, k=10, rerank_top=3, min_score=0.6)

print(f"📝 Query: {result['query']}")
print(f"📄 Sources: {', '.join(result['sources'])}")
print(f"🔍 Retrieved {result['chunks_retrieved']} chunks → Kept {result['chunks_kept']}")
print(f"📊 Relevance scores: {[f'{s:.2f}' for s in result['scores']]}")
print(f"\n🤖 Answer:\n{result['answer']}")

print("\n" + "="*60 + "\n")


🚀 COMPLETE RAG PIPELINE TEST

📝 Query: How much leave do employees get?
📄 Sources: hr_policies.docx
🔍 Retrieved 2 chunks → Kept 2
📊 Relevance scores: ['0.62', '0.60']

🤖 Answer:
[Error calling LLM: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 8.134105619s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 

In [172]:
demo_queries = [
    {
        "num": 1,
        "query": "What is the work-from-home policy?",
        "scenario": "DOCX only (HR policies)",
        "filter": None,
        "expected": "Answer from hr_policies.docx"
    },
    {
        "num": 2,
        "query": "What is CloudSync and what devices can it sync to?",
        "scenario": "CSV only (Product FAQ)",
        "filter": None,
        "expected": "Answer from product_faq.csv"
    },
    {
        "num": 3,
        "query": "What was the total revenue in Q3 2024?",
        "scenario": "TXT only (Financial summary)",
        "filter": None,
        "expected": "Answer from financial_summary.txt"
    },
    {
        "num": 4,
        "query": "Does the company offer mental health support and what products do they have?",
        "scenario": "Cross-source (HR + Product)",
        "filter": None,
        "expected": "Answers from multiple files"
    },
    {
        "num": 5,
        "query": "Tell me about the product features (answer from FAQ only)",
        "scenario": "Filtered query (CSV only)",
        "filter": "product_faq.csv",
        "expected": "Answer only from product_faq.csv"
    },
    {
        "num": 6,
        "query": "What is the color of the CEO's car?",
        "scenario": "Unanswerable (hallucination test)",
        "filter": None,
        "expected": "Should refuse to answer"
    }

]

# Run all 6 queries
print("="*80)
print("🎯 LIVE DEMO: 6 TEST QUERIES")
print("="*80)

for demo in demo_queries:
    print(f"\n\nQUERY {demo['num']}: {demo['scenario']}")
    print("="*80)
    print(f"❓ Question: {demo['query']}")
    print(f"   Expected: {demo['expected']}")

    # Run pipeline
    result = rag_pipeline_v2(
        demo['query'],
        k=10,
        rerank_top=3,
        min_score=0.6,
        source_filter=demo['filter']
    )

    print(f"\n📊 Retrieval Stats:")
    print(f"   Sources used: {', '.join(result['sources']) if result['sources'] else 'None'}")
    print(f"   Chunks retrieved → kept: {result['chunks_retrieved']} → {result['chunks_kept']}")
    print(f"   Relevance scores: {[f'{s:.3f}' for s in result['scores']]}")

    print(f"\n✅ Answer:")
    print(f"   {result['answer']}")

print("\n\n" + "="*80)
print("✓ ALL DEMO QUERIES COMPLETE")
print("="*80)


🎯 LIVE DEMO: 6 TEST QUERIES


QUERY 1: DOCX only (HR policies)
❓ Question: What is the work-from-home policy?
   Expected: Answer from hr_policies.docx

📊 Retrieval Stats:
   Sources used: hr_policies.docx
   Chunks retrieved → kept: 1 → 1
   Relevance scores: ['0.755']

✅ Answer:
   [Error calling LLM: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, mode